# Notebook Overview — Generate CLIP Text Representations

## Purpose

This notebook generates reusable CLIP text representations for NExT-QA question–answer candidates. These shared semantic representations provide the text input used by the representation-based VideoQA experiments.

For each NExT-QA annotation record, the notebook constructs five combined question–answer text inputs—one for each multiple-choice candidate answer. Each input contains the complete question followed by one candidate answer. A pretrained CLIP text encoder converts each combined text input into a normalized 512-dimensional embedding.

The generated `clip_text` artifacts are shared across the representation-based VideoQA methods. Notebook 07 combines these question–answer candidate representations with either `clip_video` or `autoencoder_video` representations for cosine-similarity scoring or learned classifier experiments, as supported by the selected configuration.

## Inputs

* Shared project configuration and constants
* NExT-QA annotation files containing questions, answer choices, and ground-truth labels
* Development-mode or full-dataset execution configuration
* Pretrained CLIP text encoder model

## Outputs

* CLIP question–answer candidate representation dataset
* CLIP text representation summary report
* Representation validation results
* Sample question–answer representation records
* Shared Google Drive artifacts when full-dataset generation is enabled

## Processing Workflow

1. Initialize the notebook environment.
2. Configure CLIP question–answer text representation generation.
3. Verify the runtime environment.
4. Prepare one combined question–answer text input for each candidate answer.
5. Load the pretrained CLIP text encoder.
6. Generate normalized CLIP question–answer candidate representations.
7. Validate the generated representation dataset.
8. Save representation artifacts locally and, in full-dataset mode, to Google Drive.
9. Generate representation summary reports.
10. Display representative question–answer representation records.
11. Summarize notebook outputs and generated artifacts.

## Downstream Consumer

Notebook 07 — Run Representation-Based VideoQA


### 🔷 Step 0 — Configure Local Constants

* Configure notebook runtime options, including GPU requirements, verbose progress reporting, and optional Google Drive write support.



In [ ]:
# ============================================================
# Step 0: Configure Local Constants
# ============================================================

# Require an NVIDIA L4 GPU.
REQUIRE_L4_GPU = True

# Display detailed notebook progress.
VERBOSE = True

# Allow generated artifacts to Google Drive.
# Disabled by default for the public tutorial.
ENABLE_GOOGLE_DRIVE_WRITES = False



### 🔷 Step 1 — Initialize Environment for CLIP Text Representation Generation

* Configure notebook execution controls for development-subset or full-dataset generation.
* Mount Google Drive and prepare the Colab execution environment.
* Clone the project repository and load shared configuration constants and utility modules.
* Verify required project paths and output directories.
* Load NExT-QA annotation metadata required for text representation generation.
* Prepare the notebook environment for CLIP-based text representation generation.


In [ ]:
# ============================================================
# Step 1: Initialize Environment for CLIP Text Representation Generation
# ============================================================

# ============================================================
# Notebook Execution Controls
# ============================================================

# False -> Generate embeddings for the development subset.
# True  -> Generate embeddings for the complete NExT-QA dataset.
# This switch controls whether the notebook performs a fast development run or
# generates the complete text-representation artifact used by later experiments.
GENERATE_FULL_DATASET = True

import os
import time
from pathlib import Path

import pandas as pd

from google.colab import drive, userdata

print("Initializing notebook environment...")
print("-" * 60)

# ------------------------------------------------------------
# DRIVE MOUNT
# ------------------------------------------------------------

# Google Drive provides persistent storage for project artifacts that must
# remain available after the temporary Colab runtime is disconnected.
GOOGLE_DRIVE_MOUNT = "/content/drive"

if not os.path.exists(GOOGLE_DRIVE_MOUNT):
    print("Mounting Google Drive...")
    drive.mount(GOOGLE_DRIVE_MOUNT)
else:
    print("Google Drive already mounted.")

# ------------------------------------------------------------
# CLONE REPOSITORY
# ------------------------------------------------------------

# Define the repository location used for project modules, dataset annotations,
# and generated output files within the current Colab session.
REPO_NAME = "videoqa-representation-comparison"
REPO_OWNER = "pgailinas"
REPO_BASE_DIR = "/content"
REPO_DIR = os.path.join(REPO_BASE_DIR, REPO_NAME)

# Retrieve the GitHub credential from Colab Secrets so authentication details
# are not stored directly in the notebook source.
github_token = userdata.get("GITHUB_TOKEN")

if github_token is None:
    raise ValueError("GITHUB_TOKEN not found in Colab Secrets.")

repo_url = (
    f"https://x-access-token:{github_token}"
    f"@github.com/{REPO_OWNER}/{REPO_NAME}.git"
)

os.chdir(REPO_BASE_DIR)

# Use a sparse checkout to retrieve only the directories required by this
# notebook, reducing repository download time and runtime storage use.
if not os.path.exists(REPO_DIR):

    print("Cloning project repository...")

    !git clone --quiet --filter=blob:none --no-checkout {repo_url}

    os.chdir(REPO_DIR)

    !git sparse-checkout init --cone
    !git sparse-checkout set src datasets outputs
    !git checkout --quiet main

else:

    # Reuse the existing repository when this initialization cell is rerun
    # during the same Colab session.
    print("Project repository already available.")
    os.chdir(REPO_DIR)

print(f"Repository ready: {REPO_DIR}")

# ------------------------------------------------------------
# LOAD PROJECT MODULES
# ------------------------------------------------------------

# Load the shared configuration first, followed by the NExT-QA metadata helpers
# used to read, combine, and summarize the annotation splits.
print("\nLoading project configuration and utility modules...")

from src.videoqa_representation_config import *
from src.nextqa_metadata import *

# ------------------------------------------------------------
# OUTPUT SETUP
# ------------------------------------------------------------

# Ensure the configured output root exists before validating the complete set
# of repository and dataset paths required by later notebook steps.
OUTPUTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

required_paths = [
    Path("src"),
    Path("datasets"),
    OUTPUTS_DIR,
    QUESTIONS_DIR,
]

missing_paths = [
    path for path in required_paths
    if not path.exists()
]

# Fail early when repository setup or dataset placement is incomplete, rather
# than allowing a later embedding-generation step to fail ambiguously.
if missing_paths:
    for path in missing_paths:
        print(f"Missing required path: {path}")

    raise FileNotFoundError(
        "One or more required project paths are missing."
    )

print("Configuration loaded.")
print("Project paths initialized.")

# ------------------------------------------------------------
# LOAD NExT-QA ANNOTATIONS
# ------------------------------------------------------------

# Load the authoritative train, validation, and test annotation files that will
# be transformed into CLIP-compatible question-and-answer text records.
print("\nLoading NExT-QA annotations...")

split_annotations = load_nextqa_split_annotations(
    annotations_dir=QUESTIONS_DIR,
    verbose=VERBOSE,
)

# Combine the split-specific tables into one dataframe while preserving split
# membership for later filtering and output generation.
annotations_df = combine_nextqa_annotations(
    split_dataframes=split_annotations,
    verbose=VERBOSE,
)

# Produce an initial dataset summary so annotation coverage can be verified
# before any text prompts or embeddings are generated.
split_summary_df = summarize_nextqa_splits(
    annotations=annotations_df,
)

print("\nDataset metadata loaded.")
print(f"Annotation records: {len(annotations_df):,}")

# Display the split-level dataset composition when detailed reporting is enabled.
if VERBOSE:
    print("\nSplit Summary")
    print("-" * 60)
    display(split_summary_df)

print("\nEnvironment initialization complete.")
print("-" * 60)
print("Notebook ready for CLIP text representation generation.")



### 🔷 Step 2 — Define CLIP Text Representation Configuration

* Define the shared CLIP text representation configuration used throughout the notebook.
* Configure development-subset or full-dataset representation generation.
* Specify the active evaluation split, development subset size, and randomization settings.
* Configure the pretrained CLIP text encoder used to generate `clip_text` representations.
* Define the representation scope as one combined question–answer text input for each candidate answer.
* Specify the question–answer text template and answer-choice columns.
* Display the active representation configuration for the current execution.



In [ ]:
# ============================================================
# Step 2: Define CLIP Text Representation Configuration
# ============================================================

print("Defining CLIP text representation configuration...")

# ------------------------------------------------------------
# CLIP text model configuration
# ------------------------------------------------------------

# Use the same pretrained CLIP model employed throughout the project so text
# representations remain compatible with the shared CLIP embedding space.
CLIP_TEXT_MODEL_NAME = "openai/clip-vit-base-patch32"

# Notebook 05 now generates one text embedding per candidate answer:
#   Question: <question>
#   Answer: <answer choice>
# This scope converts each multiple-choice question into separate
# question-and-answer candidate records for downstream comparison.
TEXT_REPRESENTATION_SCOPE = "question_answer_candidates"

# Record the semantic type of text represented by each generated embedding.
TEXT_INPUT_TYPES = [
    "question_answer",
]

# Reuse the shared annotation-column definitions so text construction remains
# aligned with the authoritative NExT-QA schema.
QUESTION_TEXT_FIELD = QUESTION_COLUMN
ANSWER_CHOICE_COLUMNS = CHOICE_COLUMNS

# Combine each question with one candidate answer in a consistent textual form
# before passing the text to the CLIP tokenizer and encoder.
QUESTION_ANSWER_TEXT_TEMPLATE = (
    "Question: {question}\n"
    "Answer: {answer_choice}"
)

# ------------------------------------------------------------
# Dataset generation configuration
# ------------------------------------------------------------

# Copy the shared dataset controls into local variables so later steps can
# clearly distinguish full-dataset generation from development evaluation.
evaluation_split = EVALUATION_SPLIT
development_subset_size = DEVELOPMENT_SUBSET_SIZE
random_seed = RANDOM_SEED

# ------------------------------------------------------------
# Display active configuration
# ------------------------------------------------------------

# Collect all active model, text-construction, dataset, and output settings into
# a compact table for reproducibility and notebook-level verification.
clip_text_config_summary = {
    "artifact_scope": "shared",
    "clip_text_model": CLIP_TEXT_MODEL_NAME,
    "representation_scope": TEXT_REPRESENTATION_SCOPE,
    "text_input_types": ", ".join(TEXT_INPUT_TYPES),
    "question_answer_text_template": QUESTION_ANSWER_TEXT_TEMPLATE,
    "generate_full_dataset": GENERATE_FULL_DATASET,
    "evaluation_split": evaluation_split,
    "development_subset_size": development_subset_size,
    "random_seed": random_seed,
    "question_text_field": QUESTION_TEXT_FIELD,
    "answer_choice_columns": ", ".join(ANSWER_CHOICE_COLUMNS),
    "output_directory": str(CLIP_TEXT_REPRESENTATIONS_DRIVE_DIR),
}

# Convert the configuration dictionary into a readable two-column dataframe
# without changing the underlying values used by later notebook steps.
clip_text_config_df = pd.DataFrame(
    clip_text_config_summary.items(),
    columns=["Configuration Item", "Value"],
)

print("CLIP text representation configuration defined.")
display(clip_text_config_df)



### 🔷 Step 3 — Verify Runtime Environment and Dependencies

* Verify the active Python runtime, operating system, and PyTorch installation.
* Confirm that CUDA is available and validate the required NVIDIA L4 GPU when enabled.
* Display detected GPU hardware and available GPU memory.
* Verify that the Hugging Face Transformers library is installed and available.
* Confirm that the required CLIP model and processor classes can be imported successfully.
* Validate that the runtime environment satisfies all software and hardware requirements.
* Display a runtime verification summary before loading the CLIP text model.

In [ ]:
# ============================================================
# Step 3: Verify Runtime Environment and Dependencies
# ============================================================

import platform
import sys

import torch

print("Verifying runtime environment...")
print("-" * 60)

# ------------------------------------------------------------
# Python
# ------------------------------------------------------------

# Record the Python interpreter and operating system so experiment results can
# be reproduced under an equivalent software environment.
print(f"Python Version : {sys.version.split()[0]}")
print(f"Platform       : {platform.platform()}")

# ------------------------------------------------------------
# PyTorch
# ------------------------------------------------------------

# Verify the installed PyTorch version that will execute the CLIP model.
print(f"PyTorch Version: {torch.__version__}")

# ------------------------------------------------------------
# CUDA
# ------------------------------------------------------------

# Confirm that GPU acceleration is available before loading the pretrained
# CLIP model and generating embeddings for the large collection of text records.
cuda_available = torch.cuda.is_available()

print(f"CUDA Available : {cuda_available}")

if REQUIRE_L4_GPU:

    # Stop immediately if the required accelerator is unavailable rather than
    # allowing a lengthy embedding-generation process to fail later.
    if not cuda_available:
        raise RuntimeError(
            "CUDA GPU is required for CLIP text representation generation."
        )

    gpu_name = torch.cuda.get_device_name(0)

    print(f"GPU            : {gpu_name}")

    # Standardize the hardware platform across experiments to improve
    # reproducibility and maintain consistent execution performance.
    if "L4" not in gpu_name:
        raise RuntimeError(
            f"NVIDIA L4 GPU required. Detected: {gpu_name}"
        )

    gpu_properties = torch.cuda.get_device_properties(0)

    gpu_memory_gb = (
        gpu_properties.total_memory
        / (1024 ** 3)
    )

    print(f"GPU Memory     : {gpu_memory_gb:.1f} GB")

# ------------------------------------------------------------
# Transformers
# ------------------------------------------------------------

# Verify that the Hugging Face Transformers library is installed because it
# provides the pretrained CLIP implementation used throughout this notebook.
try:
    import transformers

    print(f"Transformers   : {transformers.__version__}")

except ImportError:

    raise ImportError(
        "The transformers package is required."
    )

# ------------------------------------------------------------
# Verify CLIP Classes
# ------------------------------------------------------------

# Confirm that the specific CLIP model and processor classes are available
# before proceeding to model loading and text embedding generation.
try:
    from transformers import (
        CLIPModel,
        CLIPProcessor,
    )

    print("CLIP classes   : Available")

except ImportError:

    raise ImportError(
        "Unable to import CLIPModel and CLIPProcessor."
    )

# ------------------------------------------------------------
# Runtime Summary
# ------------------------------------------------------------

# Summarize the completed environment verification before transitioning to the
# CLIP model initialization and text representation generation workflow.
print("\nRuntime verification complete.")
print("-" * 60)
print("Environment is ready for CLIP text representation generation.")



### 🔷 Step 4 — Prepare CLIP Text Input Dataset

* Select either the configured development annotation subset or the complete NExT-QA annotation dataset.
* In development mode, select a reproducible sample of annotation records from the configured evaluation split.
* In full-dataset mode, include annotation records from all available dataset splits.
* Validate the required split, video, question, ground-truth answer, and answer-choice fields.
* Associate each annotation record with its ground-truth answer text.
* Construct five combined question–answer text records for each annotation—one for each candidate answer.
* Format each input using the complete question and one candidate answer.
* Retain the video identifier, question identifier, candidate index, split, ground-truth label, and representation metadata.
* Verify that each annotation produces exactly one record per answer choice.
* Display dataset statistics and representative question–answer text inputs.


In [ ]:
# ============================================================
# Step 4: Prepare CLIP Text Input Dataset
# ============================================================

import pandas as pd

print("Preparing CLIP text input dataset...")

# ------------------------------------------------------------
# Validate annotation columns
# ------------------------------------------------------------

# Confirm that the combined NExT-QA annotation table contains every field
# required to construct question-and-answer candidate text records.
required_annotation_columns = [
    "split",
    VIDEO_ID_COLUMN,
    QUESTION_COLUMN,
    GROUND_TRUTH_ANSWER_COLUMN,
    *CHOICE_COLUMNS,
]

missing_annotation_columns = [
    col for col in required_annotation_columns
    if col not in annotations_df.columns
]

# Fail before dataset selection if the authoritative annotation schema is
# incomplete or inconsistent with the shared project configuration.
if missing_annotation_columns:
    raise ValueError(
        f"annotations_df is missing required columns: {missing_annotation_columns}"
    )

# ------------------------------------------------------------
# Select annotation records
# ------------------------------------------------------------

# Use every available annotation when generating the shared full-dataset
# representation artifact.
if GENERATE_FULL_DATASET:

    eval_df = (
        annotations_df
        .copy()
        .reset_index(drop=True)
    )

else:

    # For development runs, restrict processing to the configured evaluation
    # split before selecting a reproducible subset of annotation records.
    eval_df = annotations_df[
        annotations_df["split"] == evaluation_split
    ].copy()

    if len(eval_df) == 0:
        raise ValueError(f"No records found for split: {evaluation_split}")

    sample_size = min(
        development_subset_size,
        len(eval_df),
    )

    eval_df = (
        eval_df
        .sample(
            n=sample_size,
            random_state=random_seed,
        )
        .reset_index(drop=True)
    )

# Ensure that dataset-mode selection produced at least one usable annotation.
if len(eval_df) == 0:
    raise RuntimeError("No annotation records were selected.")

# Standardize video identifiers as strings so they can be merged consistently
# with CLIP video and autoencoder representation tables in later notebooks.
eval_df[VIDEO_ID_COLUMN] = eval_df[VIDEO_ID_COLUMN].astype(str)

# Record the selected dataset splits for verification and final reporting.
input_splits = sorted(
    eval_df["split"]
    .astype(str)
    .unique()
    .tolist()
)

dataset_mode_label = (
    "full"
    if GENERATE_FULL_DATASET
    else "development"
)

print(f"Dataset mode              : {dataset_mode_label}")
print(f"Selected annotation rows  : {len(eval_df):,}")
print(f"Input splits              : {', '.join(input_splits)}")

# ------------------------------------------------------------
# Attach ground-truth answer text
# ------------------------------------------------------------

# Convert the numeric NExT-QA answer index into the corresponding answer-choice
# text so each generated candidate record retains its ground-truth reference.
def answer_index_to_text(row):
    answer_idx = int(row[GROUND_TRUTH_ANSWER_COLUMN])
    option_col = f"a{answer_idx}"

    if option_col not in CHOICE_COLUMNS:
        raise ValueError(
            f"Answer option column not valid: {option_col}"
        )

    return row[option_col]

eval_df["ground_truth_text"] = eval_df.apply(
    answer_index_to_text,
    axis=1,
)

# ------------------------------------------------------------
# Build normalized question-answer text input records
# ------------------------------------------------------------

# Expand each annotation into one text record per candidate answer so every
# answer choice receives an independent CLIP text embedding.
text_records = []

for row_index, row in eval_df.iterrows():

    video_id = str(row[VIDEO_ID_COLUMN])

    # Prefer an existing question identifier when available, while retaining a
    # deterministic row-based fallback for annotation files without one.
    raw_question_id = row.get(
        "question_id",
        row.get("qid", None),
    )

    if pd.isna(raw_question_id):
        question_id = row_index
    else:
        question_id = raw_question_id

    # Construct a unique annotation identifier that remains traceable to the
    # source split, video, question, and selected dataframe row.
    annotation_id = (
        f"{row['split']}_{video_id}_{question_id}_{row_index}"
    )

    question_text = str(row[QUESTION_COLUMN]).strip()

    if question_text == "":
        raise ValueError(
            f"Empty question text found for annotation_id: {annotation_id}"
        )

    # Pair the question with each answer option using the standardized text
    # structure expected by the CLIP tokenizer in the next processing stage.
    for choice_index, choice_col in enumerate(CHOICE_COLUMNS):

        answer_choice_text = str(row[choice_col]).strip()

        if answer_choice_text == "":
            raise ValueError(
                f"Empty answer choice text found for annotation_id "
                f"{annotation_id}, choice_index {choice_index}"
            )

        question_answer_text = (
            f"Question: {question_text}\n\n"
            f"Answer: {answer_choice_text}"
        )

        # Preserve both model input text and source metadata so embeddings can
        # later be reconstructed into grouped multiple-choice QA examples.
        text_records.append(
            {
                "record_id": f"{annotation_id}_choice_{choice_index}",
                "video": video_id,
                "question_id": question_id,
                "annotation_id": annotation_id,
                "text_type": "question_answer",
                "choice_index": choice_index,
                "text": question_answer_text,
                "question": question_text,
                "answer_choice": answer_choice_text,
                "answer": int(row[GROUND_TRUTH_ANSWER_COLUMN]),
                "ground_truth_text": row["ground_truth_text"],
                "split": row["split"],
                "representation_source": "clip_text",
            }
        )

# Convert the expanded record collection into the tabular format consumed by
# the batched CLIP text-embedding workflow.
text_input_df = pd.DataFrame(text_records)

if text_input_df.empty:
    raise RuntimeError("No CLIP text input records were generated.")

# ------------------------------------------------------------
# Validate generated text inputs
# ------------------------------------------------------------

# Validate the complete downstream schema before model inference begins.
required_text_columns = [
    "record_id",
    "video",
    "question_id",
    "annotation_id",
    "text_type",
    "choice_index",
    "text",
    "question",
    "answer_choice",
    "answer",
    "ground_truth_text",
    "split",
    "representation_source",
]

missing_text_columns = [
    col for col in required_text_columns
    if col not in text_input_df.columns
]

if missing_text_columns:
    raise ValueError(
        f"text_input_df is missing required columns: {missing_text_columns}"
    )

# Empty text would produce invalid or misleading embeddings, so verify every
# candidate contains a nonblank normalized model input.
empty_text_count = (
    text_input_df["text"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

if empty_text_count > 0:
    raise ValueError(
        f"Found {empty_text_count} empty text input records."
    )

# Record identifiers must be unique because they serve as the row-level key for
# generated embeddings and later validation.
duplicate_record_id_count = (
    text_input_df["record_id"]
    .duplicated()
    .sum()
)

if duplicate_record_id_count > 0:
    raise ValueError(
        f"Found {duplicate_record_id_count} duplicate record_id values."
    )

# Restrict text records to the single representation type defined for this
# notebook's question-and-answer candidate scope.
invalid_text_type_values = sorted(
    set(text_input_df["text_type"].unique())
    - {"question_answer"}
)

if invalid_text_type_values:
    raise ValueError(
        f"Invalid text_type values found: {invalid_text_type_values}"
    )

# Confirm that every source annotation expanded into exactly one record for
# each configured multiple-choice answer column.
records_per_annotation = (
    text_input_df
    .groupby("annotation_id")
    .size()
)

invalid_record_counts = records_per_annotation[
    records_per_annotation != len(CHOICE_COLUMNS)
]

if len(invalid_record_counts) > 0:
    raise ValueError(
        "Each annotation must have exactly one question-answer text record "
        f"per answer choice. Invalid annotation count: {len(invalid_record_counts)}"
    )

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

# Summarize the generated record types before transitioning to CLIP model
# loading and batched text representation generation.
text_type_summary_df = (
    text_input_df
    .groupby("text_type")
    .size()
    .reset_index(name="record_count")
)

print("\nCLIP text input dataset prepared successfully.")
print(f"Dataset mode       : {dataset_mode_label}")
print(f"Annotation records : {len(eval_df):,}")
print(f"Text input records : {len(text_input_df):,}")
print(f"Unique videos      : {eval_df[VIDEO_ID_COLUMN].nunique():,}")
print(f"Input splits       : {', '.join(input_splits)}")
print(f"Answer mode        : {ANSWER_MODE}")

print("\nText Type Summary:")
display(text_type_summary_df)

# Display all candidate-answer records for the first annotation so the text
# expansion structure can be inspected before embedding generation begins.
print("\nText Input Preview (First Question):")

first_annotation_id = text_input_df["annotation_id"].iloc[0]

display(
    text_input_df[
        text_input_df["annotation_id"] == first_annotation_id
    ]
)



### 🔷 Step 5 — Load CLIP Text Model

* Select the appropriate computation device for CLIP text representation generation.
* Load the pretrained CLIP text encoder model from the Hugging Face Transformers library.
* Load the corresponding CLIP processor used for text tokenization and preprocessing.
* Configure the CLIP model for inference by switching to evaluation mode.
* Verify the text embedding dimension and maximum supported text sequence length.
* Perform a sample inference to confirm successful text embedding generation.
* Display model configuration details and verification results prior to batch representation generation.

In [ ]:
# ============================================================
# Step 5: Load CLIP Text Model
# ============================================================

import torch

from transformers import (
    CLIPModel,
    CLIPProcessor,
)

print("Loading CLIP text model...")

# ------------------------------------------------------------
# Select computation device
# ------------------------------------------------------------

# Select the GPU when available so CLIP inference can be performed efficiently
# across the large question-and-answer candidate dataset.
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(f"Device: {device}")

# ------------------------------------------------------------
# Load CLIP model
# ------------------------------------------------------------

# Load the pretrained CLIP model specified in the shared notebook
# configuration so text embeddings remain consistent with the project pipeline.
clip_model = CLIPModel.from_pretrained(
    CLIP_TEXT_MODEL_NAME
)

# Move the model to the selected device and disable training-specific behavior
# because this notebook performs representation extraction only.
clip_model.to(device)
clip_model.eval()

# ------------------------------------------------------------
# Load CLIP processor
# ------------------------------------------------------------

# Load the matching processor responsible for tokenizing text into the input
# tensors expected by the pretrained CLIP text encoder.
clip_processor = CLIPProcessor.from_pretrained(
    CLIP_TEXT_MODEL_NAME
)

# ------------------------------------------------------------
# Verify model configuration
# ------------------------------------------------------------

# Read the projected CLIP embedding dimension used by downstream
# representation-based VideoQA experiments.
text_embedding_dimension = (
    clip_model.config.projection_dim
)

# Record the maximum sequence length supported by the CLIP text transformer.
text_max_position_embeddings = (
    clip_model.text_model.config.max_position_embeddings
)

print("\nCLIP text model loaded successfully.")
print(f"Model                     : {CLIP_TEXT_MODEL_NAME}")
print(f"Embedding dimension       : {text_embedding_dimension}")
print(f"Maximum text length       : {text_max_position_embeddings}")
print(f"Model device              : {device}")

# ------------------------------------------------------------
# Verify inference
# ------------------------------------------------------------

# Process a small sample string before full generation to confirm that the
# processor, model, and selected device operate together correctly.
sample_inputs = clip_processor(
    text=["CLIP model verification"],
    return_tensors="pt",
    padding=True,
)

# Move every processor output tensor to the same device as the CLIP model.
sample_inputs = {
    key: value.to(device)
    for key, value in sample_inputs.items()
}

# Disable gradient tracking because this verification performs inference only.
with torch.no_grad():
    sample_outputs = clip_model.text_model(
        **sample_inputs
    )

    # Inspect the pooled transformer output to verify the text encoder produces
    # one fixed-length feature vector for the sample input.
    sample_features = sample_outputs.pooler_output

print(
    f"Verification embedding shape : "
    f"{tuple(sample_features.shape)}"
)

# Confirm that model initialization and sample inference succeeded before
# beginning full CLIP text representation generation.
print("\nCLIP text model is ready for representation generation.")



### 🔷 Step 6 — Generate CLIP Text Representations

* Generate one CLIP text embedding for each prepared question–answer candidate record.
* Process the combined text inputs in batches to improve inference performance and GPU utilization.
* Tokenize the question–answer text using the pretrained CLIP processor.
* Encode each combined question and candidate answer using the CLIP text model and projection layer.
* Normalize each embedding vector to unit length.
* Associate the resulting embedding with its question, candidate-answer, ground-truth, split, and representation metadata.
* Store the 512-dimensional embeddings using standardized `clip_text_###` columns.
* Report the number of input records, generated embeddings, embedding dimensions, and processing time.


In [ ]:
# ============================================================
# Step 6: Generate CLIP Text Representations
# ============================================================

import time
import numpy as np
import pandas as pd
import torch
from tqdm.notebook import tqdm

print("Generating CLIP text representations...")

# Confirm that the prepared text dataset and initialized CLIP components are
# available before beginning the embedding-generation workflow.
if "text_input_df" not in globals():
    raise NameError("text_input_df was not found. Run Step 4 first.")

if "clip_model" not in globals():
    raise NameError("clip_model was not found. Run Step 5 first.")

if "clip_processor" not in globals():
    raise NameError("clip_processor was not found. Run Step 5 first.")

# ------------------------------------------------------------
# Encoding configuration
# ------------------------------------------------------------

# Keep the pretrained model in inference mode and initialize storage and timing
# for the complete batched representation-generation process.
clip_model.eval()

records = []
start_time = time.time()

# ------------------------------------------------------------
# Batch text encoding
# ------------------------------------------------------------

# Process candidate-answer text records in batches to balance GPU utilization
# and memory usage across the full NExT-QA text dataset.
for start_idx in tqdm(
    range(0, len(text_input_df), CLIP_TEXT_BATCH_SIZE),
    desc="Encoding CLIP text",
):

    batch_df = text_input_df.iloc[
        start_idx:start_idx + CLIP_TEXT_BATCH_SIZE
    ].copy()

    # Extract the normalized question-and-answer strings that will be tokenized
    # and passed through the CLIP text encoder.
    batch_texts = (
        batch_df["text"]
        .astype(str)
        .tolist()
    )

    # Tokenize the batch with padding and truncation so all records conform to
    # the fixed sequence constraints of the pretrained CLIP text transformer.
    inputs = clip_processor(
        text=batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    )

    # Move all tokenizer outputs to the same computation device as the model.
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Generate representations without retaining gradients because the model is
    # used only as a fixed pretrained feature extractor.
    with torch.no_grad():

        text_outputs = clip_model.text_model(
            **inputs
        )

        # Use the pooled transformer output as the fixed-length summary of each
        # question-and-answer candidate sequence.
        pooled_output = text_outputs.pooler_output

        # Project the pooled output into CLIP's shared semantic embedding space.
        text_features = clip_model.text_projection(
            pooled_output
        )

        # L2-normalize each projected vector so later similarity calculations
        # and learned fusion models receive consistently scaled representations.
        text_features = text_features / text_features.norm(
            dim=-1,
            keepdim=True,
        )

    # Move the generated embeddings back to CPU memory and convert them to
    # NumPy arrays for row-wise dataframe construction.
    text_features_np = (
        text_features
        .detach()
        .cpu()
        .numpy()
    )

    # Preserve all source metadata while expanding each embedding into
    # explicitly named scalar columns for downstream CSV storage.
    for row, embedding in zip(
        batch_df.to_dict("records"),
        text_features_np,
    ):

        record = dict(row)

        for i, value in enumerate(embedding):
            record[f"clip_text_{i:03d}"] = float(value)

        records.append(record)

# Measure the total encoding time for reproducibility and performance reporting.
elapsed_time = time.time() - start_time

clip_text_representation_df = pd.DataFrame(records)

if clip_text_representation_df.empty:
    raise RuntimeError(
        "No CLIP text representations were generated."
    )

# ------------------------------------------------------------
# Identify embedding columns
# ------------------------------------------------------------

# Detect the generated embedding columns from their shared naming convention so
# later validation and saving steps do not depend on a hard-coded dimension.
clip_text_columns = [
    col
    for col in clip_text_representation_df.columns
    if col.startswith("clip_text_")
]

if len(clip_text_columns) == 0:
    raise ValueError(
        "No CLIP text embedding columns were generated."
    )

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

# Retain the active generation mode in the final report so development and
# full-dataset artifacts can be distinguished clearly.
dataset_mode_label = (
    "full"
    if GENERATE_FULL_DATASET
    else "development"
)

print("\nCLIP text representation generation complete.")
print(f"Dataset mode         : {dataset_mode_label}")
print(f"Input text records   : {len(text_input_df):,}")
print(f"Embedding records    : {len(clip_text_representation_df):,}")
print(f"Embedding dimensions : {len(clip_text_columns):,}")
print(f"Batch size           : {CLIP_TEXT_BATCH_SIZE}")
print(f"Elapsed time         : {elapsed_time:.1f} seconds")
print(
    f"Average/record       : "
    f"{elapsed_time / len(clip_text_representation_df):.3f} seconds"
)



### 🔷 Step 7 — Validate CLIP Text Representation Dataset

* Verify that the CLIP question–answer representation dataset was generated successfully.
* Confirm that every representation record has the `question_answer` text type.
* Validate the total number of representation records, unique videos, and unique annotation records.
* Verify that each annotation contains exactly one representation record for every candidate answer.
* Confirm that the expected CLIP text embedding dimensionality was produced.
* Check that all embedding values are present and numeric.
* Detect duplicate representation identifiers.
* Display a validation summary describing the integrity of the generated representation dataset.
* Confirm readiness for downstream representation-based VideoQA experiments.


In [ ]:
# ============================================================
# Step 7: Validate CLIP Text Representation Dataset
# ============================================================

import pandas as pd

print("Validating CLIP text representation dataset...")

# Confirm that the generated representation dataframe and embedding-column
# definitions are available before beginning dataset-level validation.
if "clip_text_representation_df" not in globals():
    raise NameError(
        "clip_text_representation_df was not found. Run Step 6 first."
    )

if "clip_text_columns" not in globals():
    raise NameError(
        "clip_text_columns was not found. Run Step 6 first."
    )

# ------------------------------------------------------------
# Validation Summary
# ------------------------------------------------------------

# Collect structural, coverage, datatype, and uniqueness checks into one
# summary so the generated artifact can be reviewed before it is saved.
validation_summary = {
    "representation_records": len(
        clip_text_representation_df
    ),
    "question_answer_records": (
        clip_text_representation_df["text_type"]
        .eq("question_answer")
        .sum()
    ),
    "unique_videos": (
        clip_text_representation_df["video"]
        .nunique()
    ),
    "unique_annotation_records": (
        clip_text_representation_df["annotation_id"]
        .nunique()
    ),
    "embedding_dimensions": len(
        clip_text_columns
    ),
    "missing_embedding_values": (
        clip_text_representation_df[
            clip_text_columns
        ]
        .isna()
        .sum()
        .sum()
    ),
    "non_numeric_embedding_columns": sum(
        not pd.api.types.is_numeric_dtype(
            clip_text_representation_df[col]
        )
        for col in clip_text_columns
    ),
    "duplicate_record_ids": (
        clip_text_representation_df["record_id"]
        .duplicated()
        .sum()
    ),
}

# Convert the validation results into a readable table for notebook inspection.
validation_df = pd.DataFrame(
    validation_summary.items(),
    columns=["Validation Check", "Value"],
)

display(validation_df)

# ------------------------------------------------------------
# Validation Checks
# ------------------------------------------------------------

# Verify that every generated row represents a question paired with one
# candidate answer, matching the representation scope defined in Step 2.
if validation_summary["question_answer_records"] != validation_summary["representation_records"]:
    raise ValueError(
        "All CLIP text representation records must have "
        "text_type='question_answer'."
    )

# Confirm that each source annotation produced exactly one representation for
# every configured answer choice.
records_per_annotation = (
    clip_text_representation_df
    .groupby("annotation_id")
    .size()
)

invalid_record_counts = records_per_annotation[
    records_per_annotation != len(CHOICE_COLUMNS)
]

if len(invalid_record_counts) > 0:
    raise ValueError(
        "Each annotation must have exactly one question-answer "
        "representation record per answer choice. "
        f"Invalid annotation count: {len(invalid_record_counts)}"
    )

# Missing embedding values would make downstream similarity and fusion
# calculations incomplete or invalid.
if validation_summary["missing_embedding_values"] > 0:
    raise ValueError(
        "Missing values detected in CLIP text representations."
    )

# Every embedding dimension must remain numeric so vectors can be converted
# directly into tensors or NumPy arrays in Notebook 07.
if validation_summary["non_numeric_embedding_columns"] > 0:
    raise ValueError(
        "Non-numeric embedding columns detected."
    )

# Record identifiers serve as row-level keys, so duplicates could create
# ambiguous candidate mappings during downstream joins.
if validation_summary["duplicate_record_ids"] > 0:
    raise ValueError(
        "Duplicate record IDs detected."
    )

# Confirm that the text representation artifact satisfies the structural and
# numerical requirements for downstream representation-based VideoQA.
print(
    "\nRepresentation validation passed. "
    "All records are question-answer representations with no missing, "
    "non-numeric, or duplicate records detected."
)



### 🔷 Step 8 — Save CLIP Text Representation Files

* Create the local output directory when necessary.
* Save the generated CLIP text representation dataset to local project storage.
* Verify that the representation dataset was written successfully.
* If full-dataset generation and Google Drive writes are enabled, copy the representation dataset to the shared Google Drive output directory.
* Otherwise, retain the representation dataset in local storage without overwriting the shared Drive artifact.
* Report the number of generated representation records and embedding dimensions.
* Display the applicable output locations for downstream representation-based VideoQA experiments.




In [ ]:
# ============================================================
# Step 8: Save CLIP Text Representation Files
# ============================================================

import shutil

print("Saving CLIP text representation files...")

# Confirm that the generated representation dataframe and identified embedding
# columns are available before writing any output artifacts.
if "clip_text_representation_df" not in globals():
    raise NameError(
        "clip_text_representation_df was not found. Run Step 6 first."
    )

if "clip_text_columns" not in globals():
    raise NameError(
        "clip_text_columns was not found. Run Step 6 first."
    )

# ------------------------------------------------------------
# Determine dataset mode
# ------------------------------------------------------------

# Label the active run so development outputs can be distinguished from the
# authoritative full-dataset representation artifact.
dataset_mode_label = (
    "full"
    if GENERATE_FULL_DATASET
    else "development"
)

# ------------------------------------------------------------
# Create local output directory
# ------------------------------------------------------------

# Ensure the repository-local output directory exists before saving the
# generated representation table.
CLIP_TEXT_LOCAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# Save representation dataset locally
# ------------------------------------------------------------

# Save the complete metadata and embedding table as a CSV file that can be
# inspected locally and consumed by later representation-based notebooks.
clip_text_representation_df.to_csv(
    CLIP_TEXT_REPRESENTATIONS_LOCAL_CSV,
    index=False,
)

# ------------------------------------------------------------
# Verify local output
# ------------------------------------------------------------

# Verify that the local artifact was created successfully before attempting any
# optional persistent copy to Google Drive.
if not CLIP_TEXT_REPRESENTATIONS_LOCAL_CSV.exists():
    raise FileNotFoundError(
        f"Failed to create local file: "
        f"{CLIP_TEXT_REPRESENTATIONS_LOCAL_CSV}"
    )

# ------------------------------------------------------------
# Copy full-dataset artifact to Google Drive
# ------------------------------------------------------------

# Track whether the persistent shared artifact was written so the final summary
# can report the output behavior clearly.
drive_artifact_written = False

if GENERATE_FULL_DATASET and ENABLE_GOOGLE_DRIVE_WRITES:

    # Only full-dataset runs may update the shared Google Drive artifact used by
    # downstream notebooks and future Colab sessions.
    CLIP_TEXT_REPRESENTATIONS_DRIVE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        CLIP_TEXT_REPRESENTATIONS_LOCAL_CSV,
        CLIP_TEXT_REPRESENTATIONS_DRIVE_CSV,
    )

    if not CLIP_TEXT_REPRESENTATIONS_DRIVE_CSV.exists():
        raise FileNotFoundError(
            f"Failed to create Drive file: "
            f"{CLIP_TEXT_REPRESENTATIONS_DRIVE_CSV}"
        )

    drive_artifact_written = True

elif GENERATE_FULL_DATASET:

    # Retain the locally verified artifact when Google Drive writes are disabled.
    print("Google Drive writes are disabled.")
    print(
        "The full-dataset representation file remains in local storage."
    )

else:

    # Preserve the authoritative shared representation file when running a
    # smaller development experiment.
    print(
        "Development mode complete. Local representation file was "
        "created, but the persistent shared Drive artifact was not "
        "overwritten."
    )

# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

# Report the saved locations and dataset dimensions before the artifact is used
# by Notebook 07 for representation-based VideoQA experiments.
print("\nCLIP text representation dataset saved successfully.")
print(f"Dataset mode              : {dataset_mode_label}")
print(f"Local output file         : {CLIP_TEXT_REPRESENTATIONS_LOCAL_CSV}")

if drive_artifact_written:
    print(f"Drive output file         : {CLIP_TEXT_REPRESENTATIONS_DRIVE_CSV}")
elif GENERATE_FULL_DATASET:
    print("Drive output file         : not written because Google Drive writes are disabled")
else:
    print("Drive output file         : not written in development mode")

print(f"Representation rows       : {len(clip_text_representation_df):,}")
print(f"Embedding dimensions      : {len(clip_text_columns):,}")
print(
    f"Question-answer records   : "
    f"{(clip_text_representation_df['text_type'] == 'question_answer').sum():,}"
)



### 🔷 Step 9 — Generate CLIP Text Representation Summary Report

* Generate summary statistics describing the completed CLIP question–answer representation dataset.
* Report the dataset mode and dataset splits included in the generated artifact.
* Record the CLIP model, representation scope, answer mode, and batch size.
* Summarize the total number of question–answer candidate records, unique videos, and unique annotation records.
* Report the number of answer choices represented for each question.
* Verify the CLIP text embedding dimensionality and check for missing embedding values.
* Save the summary report to local project storage.
* If full-dataset generation and Google Drive writes are enabled, copy the summary report to the shared Google Drive representation directory.
* Otherwise, retain the summary report in local storage without overwriting the shared Drive artifact.
* Display the completed summary and applicable output locations for verification.




In [ ]:
# ============================================================
# Step 9: Generate CLIP Text Representation Summary Report
# ============================================================

import shutil
import pandas as pd

print("Generating CLIP text representation summary report...")

# Confirm that the generated representation dataframe and embedding-column
# definitions are available before calculating the final artifact summary.
if "clip_text_representation_df" not in globals():
    raise NameError(
        "clip_text_representation_df was not found. Run Step 6 first."
    )

if "clip_text_columns" not in globals():
    raise NameError(
        "clip_text_columns was not found. Run Step 6 first."
    )

# ------------------------------------------------------------
# Determine dataset mode
# ------------------------------------------------------------

# Label the report according to whether it describes the authoritative
# full-dataset artifact or a smaller development run.
dataset_mode_label = (
    "full"
    if GENERATE_FULL_DATASET
    else "development"
)

# Record every dataset split represented in the generated text artifact.
input_splits = sorted(
    clip_text_representation_df["split"]
    .astype(str)
    .unique()
    .tolist()
)

# ------------------------------------------------------------
# Compute summary statistics
# ------------------------------------------------------------

# Count the generated question-and-answer candidate representations.
question_answer_record_count = (
    clip_text_representation_df["text_type"]
    .eq("question_answer")
    .sum()
)

# Count the unique source annotations before their expansion into one record
# per multiple-choice answer candidate.
unique_annotation_records = (
    clip_text_representation_df["annotation_id"]
    .nunique()
)

# Verify and report whether any generated embedding dimensions contain
# missing values.
missing_embedding_values = (
    clip_text_representation_df[clip_text_columns]
    .isna()
    .sum()
    .sum()
)

# Assemble a compact machine-readable description of the representation
# artifact, model configuration, dataset coverage, and validation status.
summary_rows = [
    {"metric": "artifact_scope", "value": "shared"},
    {"metric": "dataset_mode", "value": dataset_mode_label},
    {"metric": "input_splits", "value": ", ".join(input_splits)},
    {"metric": "representation_type", "value": "clip_question_answer_text_representation"},
    {"metric": "clip_text_model", "value": CLIP_TEXT_MODEL_NAME},
    {"metric": "clip_text_batch_size", "value": CLIP_TEXT_BATCH_SIZE},
    {"metric": "representation_scope", "value": CLIP_TEXT_REPRESENTATION_SCOPE},
    {"metric": "answer_mode", "value": ANSWER_MODE},
    {"metric": "representation_records", "value": len(clip_text_representation_df)},
    {"metric": "question_answer_records", "value": int(question_answer_record_count)},
    {"metric": "unique_videos", "value": clip_text_representation_df["video"].nunique()},
    {"metric": "unique_annotation_records", "value": int(unique_annotation_records)},
    {"metric": "answer_choices_per_question", "value": len(CHOICE_COLUMNS)},
    {"metric": "embedding_dimensions", "value": len(clip_text_columns)},
    {"metric": "missing_embedding_values", "value": int(missing_embedding_values)},
]

clip_text_summary_df = pd.DataFrame(summary_rows)

# ------------------------------------------------------------
# Save summary report locally
# ------------------------------------------------------------

# Ensure the repository-local output directory exists before saving the
# companion summary CSV.
CLIP_TEXT_LOCAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Save the summary separately from the large representation table so later
# notebooks and documentation can inspect artifact metadata efficiently.
clip_text_summary_df.to_csv(
    CLIP_TEXT_SUMMARY_LOCAL_CSV,
    index=False,
)

if not CLIP_TEXT_SUMMARY_LOCAL_CSV.exists():
    raise FileNotFoundError(
        f"Failed to create local summary file: "
        f"{CLIP_TEXT_SUMMARY_LOCAL_CSV}"
    )

# ------------------------------------------------------------
# Copy full-dataset summary artifact to Google Drive
# ------------------------------------------------------------

# Track whether the persistent summary artifact was written for final reporting.
summary_drive_artifact_written = False

if GENERATE_FULL_DATASET and ENABLE_GOOGLE_DRIVE_WRITES:

    # Full-dataset runs update the shared Google Drive summary alongside the
    # persistent representation dataset saved in Step 8.
    CLIP_TEXT_REPRESENTATIONS_DRIVE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        CLIP_TEXT_SUMMARY_LOCAL_CSV,
        CLIP_TEXT_SUMMARY_DRIVE_CSV,
    )

    if not CLIP_TEXT_SUMMARY_DRIVE_CSV.exists():
        raise FileNotFoundError(
            f"Failed to create Drive summary file: "
            f"{CLIP_TEXT_SUMMARY_DRIVE_CSV}"
        )

    summary_drive_artifact_written = True

elif GENERATE_FULL_DATASET:

    # Full-dataset runs retain their local summary when Google Drive writes
    # are disabled.
    print("Google Drive writes are disabled.")
    print("The full-dataset summary report remains in local storage.")

else:

    # Development runs retain their local summary without replacing the
    # authoritative shared full-dataset report.
    print(
        "Development mode complete. Local summary report was "
        "created, but the persistent shared Drive summary was not "
        "overwritten."
    )

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

# Report the saved summary locations and display the final artifact metadata
# before concluding the notebook workflow.
print("CLIP question-answer text representation summary report saved.")
print(f"Dataset mode       : {dataset_mode_label}")
print(f"Local summary file : {CLIP_TEXT_SUMMARY_LOCAL_CSV}")

if summary_drive_artifact_written:
    print(f"Drive summary file : {CLIP_TEXT_SUMMARY_DRIVE_CSV}")
elif GENERATE_FULL_DATASET:
    print("Drive summary file : not written because Google Drive writes are disabled")
else:
    print("Drive summary file : not written in development mode")

display(clip_text_summary_df)



### 🔷 Step 10 — Display Sample CLIP Text Representation Records

* Randomly select representative CLIP question–answer candidate records for inspection.
* Display the combined question–answer text, candidate index, answer metadata, ground-truth text, representation source, and selected embedding values.
* Confirm that one combined representation was generated for each candidate answer.
* Report the total representation count, unique videos, unique annotation records, answer choices per question, and embedding dimensionality.
* Provide a qualitative sanity check before downstream representation-based VideoQA experiments.



In [ ]:
# ============================================================
# Step 10: Display Sample CLIP Text Representation Records
# ============================================================

import pandas as pd

print("Displaying sample CLIP question-answer text representation records...")

# Confirm that the generated representation dataframe and embedding-column
# definitions are available before selecting and displaying sample records.
if "clip_text_representation_df" not in globals():
    raise NameError(
        "clip_text_representation_df was not found. Run Step 6 first."
    )

if "clip_text_columns" not in globals():
    raise NameError(
        "clip_text_columns was not found. Run Step 6 first."
    )

# ------------------------------------------------------------
# Select sample records
# ------------------------------------------------------------

# Limit the preview to at most ten records so the notebook remains readable
# while still showing representative metadata and embedding values.
sample_count = min(
    10,
    len(clip_text_representation_df),
)

# Use the shared random seed so the displayed sample is reproducible across
# repeated notebook runs.
sample_text_representation_df = (
    clip_text_representation_df
    .sample(
        n=sample_count,
        random_state=RANDOM_SEED,
    )
    .reset_index(drop=True)
)

# Display key source fields together with only the first five embedding
# dimensions rather than the full high-dimensional CLIP vector.
display_columns = [
    "record_id",
    "video",
    "question_id",
    "text_type",
    "choice_index",
    "text",
    "answer",
    "ground_truth_text",
    "representation_source",
    *clip_text_columns[:5],
]

print(f"Displaying {sample_count} CLIP question-answer text representation records...")

# Identify whether the displayed artifact was produced from the full dataset or
# from a smaller development subset.
dataset_mode_label = (
    "full"
    if GENERATE_FULL_DATASET
    else "development"
)

print(f"Dataset mode          : {dataset_mode_label}")
print(
    f"Input splits          : "
    f"{', '.join(sorted(clip_text_representation_df['split'].astype(str).unique()))}"
)
print(f"CLIP model            : {CLIP_TEXT_MODEL_NAME}")
print(f"Representation source : clip_text")
print(f"Embedding dimensions  : {len(clip_text_columns)}")
print(f"Showing {sample_count} sample records with the first 5 embedding columns.")

# Present the selected records for visual inspection of candidate text,
# identifiers, labels, and representative embedding values.
display(
    sample_text_representation_df[
        display_columns
    ]
)

# ------------------------------------------------------------
# Representation Summary
# ------------------------------------------------------------

# Count all question-and-answer candidate rows in the completed representation
# dataset.
question_answer_record_count = (
    clip_text_representation_df["text_type"]
    .eq("question_answer")
    .sum()
)

# Count the unique source annotations before their expansion into one row per
# answer choice.
unique_annotation_records = (
    clip_text_representation_df["annotation_id"]
    .nunique()
)

# Conclude the notebook with a concise summary of representation coverage and
# dimensionality before these embeddings are consumed by Notebook 07.
print("\nCLIP Question-Answer Text Representation Summary")
print("-" * 60)
print(f"Representation records       : {len(clip_text_representation_df):,}")
print(f"Question-answer records      : {question_answer_record_count:,}")
print(f"Unique videos                : {clip_text_representation_df['video'].nunique():,}")
print(f"Unique annotation records    : {unique_annotation_records:,}")
print(f"Answer choices per question  : {len(CHOICE_COLUMNS):,}")
print(f"Embedding dimensions         : {len(clip_text_columns):,}")



### 🔷 Step 11 — Notebook Summary

* Summarize the completed CLIP question–answer text representation workflow.
* Report the artifact scope, dataset mode, included dataset splits, CLIP model, and development subset configuration.
* Summarize the number of question–answer candidate records, unique videos, unique annotations, answer choices per question, and embedding dimensions.
* List the locally generated representation and summary artifacts.
* Identify whether the persistent shared Google Drive artifacts were written.
* Confirm that the generated `clip_text` representations are ready for downstream representation-based VideoQA experiments in Notebook 07.


In [ ]:
# ============================================================
# Step 11: Notebook Summary
# ============================================================

print("Notebook 05 complete.")
print("=" * 60)

# Identify whether the completed notebook processed the authoritative full
# dataset or a smaller development subset.
dataset_mode_label = (
    "full"
    if GENERATE_FULL_DATASET
    else "development"
)

# Collect the dataset splits represented in the generated text artifact for the
# final configuration summary.
input_splits = sorted(
    clip_text_representation_df["split"]
    .astype(str)
    .unique()
    .tolist()
)

# Summarize the shared configuration used to generate the CLIP text
# representations.
print("\nCLIP Question-Answer Text Representations — Shared Configuration")
print("-" * 60)
print("Artifact scope           : shared")
print(f"Dataset mode             : {dataset_mode_label}")
print(f"CLIP text model          : {CLIP_TEXT_MODEL_NAME}")
print(f"Input splits             : {', '.join(input_splits)}")

# Report the configured subset size only when the notebook was run in
# development mode.
if GENERATE_FULL_DATASET:
    print("Development subset size  : not applicable")
else:
    print(f"Development subset size  : {DEVELOPMENT_SUBSET_SIZE}")

print(f"QA format                : {ANSWER_MODE}")

# Report the size and structure of the completed representation dataset.
print("\nRepresentation Dataset Summary")
print("-" * 60)
print(f"Representation records   : {len(clip_text_representation_df):,}")

print(
    f"Question-answer records  : "
    f"{(clip_text_representation_df['text_type'] == 'question_answer').sum():,}"
)

print(f"Unique videos            : {clip_text_representation_df['video'].nunique():,}")

# Count the original annotation records before each annotation was expanded
# into one representation row per answer choice.
unique_annotation_records = (
    clip_text_representation_df["annotation_id"]
    .nunique()
)

print(f"Unique annotation records: {unique_annotation_records:,}")
print(f"Answer choices/question  : {len(CHOICE_COLUMNS):,}")
print(f"Embedding dimensions     : {len(clip_text_columns):,}")

# List the local and persistent artifacts produced by the notebook.
print("\nShared Representation Outputs")
print("-" * 60)
print(f"Local text artifact      : {CLIP_TEXT_REPRESENTATIONS_LOCAL_CSV}")
print(f"Local summary artifact   : {CLIP_TEXT_SUMMARY_LOCAL_CSV}")

# Full-dataset runs update the shared Google Drive artifacts, while development
# runs preserve the existing authoritative files.
if GENERATE_FULL_DATASET:
    print(f"Shared text artifact     : {CLIP_TEXT_REPRESENTATIONS_DRIVE_CSV}")
    print(f"Shared summary artifact  : {CLIP_TEXT_SUMMARY_DRIVE_CSV}")
else:
    print("Shared text artifact     : not written in development mode")
    print("Shared summary artifact  : not written in development mode")

print(f"Shared output directory  : {CLIP_TEXT_REPRESENTATIONS_DRIVE_DIR}")

# Conclude with a concise checklist of the notebook outputs and their readiness
# for downstream representation-based VideoQA experiments.
print("\nNotebook 05 generated:")
print("- CLIP question-answer text representation dataset")
print("- CLIP question-answer text representation summary")
print("- Validated question-answer representation records")
print("- Sample question-answer representation records")

print("\nNotebook 05 outputs are ready for downstream")
print("representation-based VideoQA experiments.")

